In [ ]:
# pip install accelerate
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large", device_map="auto")



s:\ML-project\GEN-AI-H2S-notebooks-\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
s:\ML-project\GEN-AI-H2S-notebooks-\venv\Lib\site-packages\accelerate\utils\modeling.py:1614: UserWarning: The following device_map keys do not match any submodules in the model: ['decoder.embed_tokens']
  warnings.warn(
Some parameters are on the meta device because they w

<pad> Wie alte sind Sie?</s>


In [17]:
input_text = "CLAIM The sky is green. VERDICT FALSE . EVIDENCE Rayleigh scattering of sunlight in Earth's atmosphere causes diffuse sky radiation, which is why the sky appears blue."
 
input_ids = tokenizer(input_text, return_tensors="pt",truncation=True).input_ids.to("cuda")

outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0]))


<pad> The sky is blue.</s>


In [23]:
tokenizer.convert_ids_to_tokens(outputs[0])
tokenizer.decode(outputs[0],skip_special_tokens=True)

'The sky is blue.'

In [14]:
def summarize_claim(claim, evidence_chunks,ver, max_input_len=1024, max_output_len=150):
    # Prepare input text
    input_text = f"""
    Claim: {claim}
    Verdict: {ver}
    Evidence:
    {"\n\n---\n\n".join(evidence_chunks)}

    Write a short explanation for why the verdict is {ver}.
    """

    
    # Tokenize
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=max_input_len).to('cuda')
    
    # Generate summary
    summary_ids = model.generate(inputs["input_ids"], max_length=max_output_len, num_beams=4, early_stopping=True)
    
    # Decode
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

claim = 'The sky is green.'
ver  ='FALSE'
evi = ["EVIDENCE Rayleigh scattering of sunlight in Earth's atmosphere causes diffuse sky radiation, which is why the sky appears blue."]

f=summarize_claim(claim=claim,evidence_chunks=evi,ver=ver)
f

"Rayleigh scattering of sunlight in Earth's atmosphere causes diffuse sky radiation, which is why the sky appears blue."

In [15]:
retrieved_docs = [
    "Nikola Tesla was an inventor known for his work on AC electricity.", # Less relevant
    "Tesla was born to Serbian parents in the village of Smiljan, Austrian Empire (modern-day Croatia).", # Highly relevant
    "Thomas Edison was a rival of Tesla." # Not relevant
]

user_claim = "Nikola Tesla was born in Croatia."

ver = "TRUE"

a  = summarize_claim(claim=user_claim,evidence_chunks=retrieved_docs,ver=ver)
a

'Nikola Tesla was born in the village of Smiljan, Austrian Empire (modern-day Croatia).'

In [24]:
import torch

class summarizer:
    def __init__(self):
        self.model_name = "google/flan-t5-large"
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(self.device)
        try:
            self.model =  T5ForConditionalGeneration.from_pretrained(self.model_name, device_map="auto")
            self.tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
            self.model.to(self.device)
        except Exception as e:
            raise RuntimeError(f"Could not fetch model and tokenizer from HF | {e}")
        
    def forward(self,claim,top_evidence,verdict, max_input_len = 1024,max_output_len = 150):

        evidences = [e[1] for e in top_evidence]

        if not evidences:
            raise ValueError("No evidence provided")
        
        input_text = f"""
        Claim: {claim}
        Verdict: {verdict}
        Evidence:
        {"\n\n---\n\n".join(top_evidence)}

        Write a short explanation for why the verdict is {ver}.
                    """

    
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=max_input_len).to('cuda')
        
        summary_ids = model.generate(inputs["input_ids"], max_length=max_output_len, num_beams=4, early_stopping=True)
        
        summary =  tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        return verdict,summary
    
    def __call__(self,claim,top_evidence,verdict, max_input_len = 1024,max_output_len = 150):
        return self.forward(self,claim,top_evidence,verdict, max_input_len = 1024,max_output_len = 150)
        